# AI Tutor — **mt50 cloud-API legs on Colab** · groks + gemini-2.5-pro re-run

Runs the remaining CLOUD-API rows of the mt50 board (--multi-turn --sample
50 --seed 5) on Colab so they proceed **in parallel** with the local
sweep (which finishes the glms). **No GPU and no Ollama** — the tutor is an API
(Vertex Model Garden for grok, Google API for gemini-2.5-pro); a plain **CPU
runtime** is correct here.

**Two tabs, one group each (Cell 7b dropdown):**
- **groks** — grok-4.1-fast-reasoning / -non-reasoning, grok-4.20-reasoning /
  -non-reasoning (~2-4h each, sequential; ~10-14h for the tab)
- **gemini** — the gemini-2.5-pro re-run (its local first attempt hung at ~75%
  and was killed; ~5-8h)

Results write to Drive `ai-tutor-eval-multiturn/mt50/` — the same folder the OSS legs
used. **Make sure only ONE `mt50` folder exists in Drive** (two tabs once
raced and created duplicates); both tabs here share it safely because they
score different models. Download into the repo's `offline_eval/multi_turn_results/mt50/` to merge with the
local legs.

**Before you start** — Colab Secrets (🔑, *Notebook access ON*):
- `GH_TOKEN` — GitHub classic PAT, `repo` scope.
- `ANTHROPIC_API_KEY` — required (student-sim + judge).
- `GOOGLE_API_KEY` + `OPENAI_API_KEY` — gemini tutor calls + grader cascade.

**Vertex auth (grok tab)**: Cell 5b pops the Google login — use the account
with access to GCP project `ai-tutor-499714` (pixeldesignlabs). The gemini tab
needs only the API key, but running 5b is harmless there.

## Cell 1 — mount Drive (CPU runtime is fine — no GPU needed)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone the repo (branch `pixeldesignlabs-dev-portuguese`)

In [ ]:
from google.colab import userdata
import subprocess, os
os.chdir('/content')   # a re-run's cwd may be the about-to-be-deleted clone
tok = (userdata.get('GH_TOKEN') or '').strip()
assert tok and ' ' not in tok, "GH_TOKEN missing or contains a space — re-save the secret"
url = f"https://{tok}@github.com/eai6/ai-tutor.git"
subprocess.run(['rm', '-rf', '/content/ai-tutor'], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '-b', 'pixeldesignlabs-dev-portuguese', url, '/content/ai-tutor'], check=True)
os.chdir('/content/ai-tutor')
print('cloned at', os.getcwd())

## Cell 3 — fix hardcoded laptop paths (essential)

In [ ]:
!sed -i 's#/home/daniel/Documents/work/Nyansapo/web/ai-tutor#/content/ai-tutor#g; s#\$ROOT/venv/bin/python#python#g; s#venv/bin/python#python#g' offline_eval/*.py offline_eval/*.sh

## Cell 4 — install deps (a few min; ignore pip resolver warnings)

In [ ]:
!pip install -q -r requirements.txt

## Cell 5 — write .env from Colab Secrets (+ GCP project for Vertex)

In [ ]:
from google.colab import userdata
open('.env', 'w').write(
    "SECRET_KEY=colab-eval\nDEBUG=True\nEMBEDDING_BACKEND=sqlite\n"
    f"ANTHROPIC_API_KEY={userdata.get('ANTHROPIC_API_KEY')}\n"
    f"GOOGLE_API_KEY={userdata.get('GOOGLE_API_KEY')}\n"
    f"OPENAI_API_KEY={userdata.get('OPENAI_API_KEY')}\n"
    "GOOGLE_CLOUD_PROJECT=ai-tutor-499714\n"
    "GOOGLE_CLOUD_LOCATION=global\n")
print('.env written')

## Cell 5b — Vertex auth (needed for the **groks** tab)
Pops the Google sign-in; pick the account with access to `ai-tutor-499714`. Sets up ADC that google-auth (VertexModelGardenClient) picks up.

In [ ]:
from google.colab import auth
auth.authenticate_user(project_id='ai-tutor-499714')
print('ADC ready for', 'ai-tutor-499714')

## Cell 6 — fresh DB + eval fixtures

In [ ]:
!python manage.py migrate
!python manage.py loaddata evals/fixtures/institution.json evals/fixtures/lessons.json

## Cell 7 — persist results to Drive (symlink → survives disconnects)

In [ ]:
!mkdir -p /content/drive/MyDrive/ai-tutor-eval-multiturn/mt50
!rm -rf offline_eval/multi_turn_results/mt50 && mkdir -p offline_eval/multi_turn_results && ln -s /content/drive/MyDrive/ai-tutor-eval-multiturn/mt50 offline_eval/multi_turn_results/mt50
import os, glob
print('this run writes to:', os.path.realpath('offline_eval/multi_turn_results/mt50'))
done = sorted(os.path.basename(p)[:-5] for p in glob.glob('offline_eval/multi_turn_results/mt50/*.json'))
print('already scored on Drive:', done or '(none yet)')

## Cell 7b — **pick this tab's GROUP**
- **groks** — the four grok MaaS rows (needs Cell 5b auth)
- **gemini** — the gemini-2.5-pro re-run (API key only)

In [ ]:
GROUP = "groks"  #@param ["groks", "gemini"]
print("This tab will evaluate group:", GROUP)

## Cell 8 — write THIS tab's model list

In [ ]:
GROUPS = {
    "groks": [
        [
            "vertex_model_garden/xai/grok-4.1-fast-reasoning",
            "grok-4.1-fast-reasoning",
            "global"
        ],
        [
            "vertex_model_garden/xai/grok-4.1-fast-non-reasoning",
            "grok-4.1-fast-non-reasoning",
            "global"
        ],
        [
            "vertex_model_garden/xai/grok-4.20-reasoning",
            "grok-4.20-reasoning",
            "global"
        ],
        [
            "vertex_model_garden/xai/grok-4.20-non-reasoning",
            "grok-4.20-non-reasoning",
            "global"
        ]
    ],
    "gemini": [
        [
            "google/gemini-2.5-pro",
            "gemini-2.5-pro",
            ""
        ]
    ]
}
GROUP = globals().get('GROUP')
assert GROUP in GROUPS, "Run Cell 7b first."
lines = ["# mt50 cloud-API legs — " + GROUP + " (this tab)"]
for spec, safe, region in GROUPS[GROUP]:
    lines.append(f"{spec:<60} {safe:<28} {region}".rstrip())
open('offline_eval/cloud_models_colab.txt', 'w').write("\n".join(lines) + "\n")
print(open('offline_eval/cloud_models_colab.txt').read())

## Cell 9 — run the sweep (resume-safe; ~2-4h per grok, ~5-8h for 2.5-pro)
Console stays quiet per model until it saves — progress streams to the per-model log in `offline_eval/multi_turn_results/mt50/`. Check liveness anytime with:
```
!tail -3 offline_eval/multi_turn_results/mt50/<model>.log
```

In [ ]:
!RESULTS_DIR=$PWD/offline_eval/multi_turn_results/mt50 SIMPLE_TUTOR_ENGINE=1 \
  CLOUD_MODELS_FILE=$PWD/offline_eval/cloud_models_colab.txt \
  MODE="--multi-turn --sample 50 --seed 5" bash offline_eval/run_cloud.sh

## Cell 10 — board + end-reasons for whatever is on Drive

In [ ]:
import json, glob, os
from collections import Counter
print(f"{'MODEL':<30}{'PASS':>8}   END-REASONS")
print('-' * 75)
for f in sorted(glob.glob('offline_eval/multi_turn_results/mt50/*.json')):
    name = os.path.basename(f)[:-5]
    if name.startswith('_'):
        continue
    d = json.load(open(f))
    ends = Counter(r.get('sim_reason') or ('errored' if r.get('error') else '?')
                   for r in d['results'])
    reasons = '  '.join(f"{k}:{v}" for k, v in ends.most_common(3))
    print(f"{name:<30}{d['passed']:>4}/{d['total_scenarios']:<3} {reasons}")